# 01 · EDA — dataset

Goal: know the class balance, box sizes and camera conditions **before** training.

> Rule: functions used 2+ times move to `src/vehicle_detect/`. This notebook never writes to `data/processed/` — run `make prepare` / `dvc repro` instead.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

# run from the project root so configs/ and data/ paths work from any notebook
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from vehicle_detect.config import load_config
from vehicle_detect.data.mapping import class_histogram

cfg = load_config("configs/data/thai_traffic.yaml")
ann = json.loads(Path("data/processed/coco/train/_annotations.coco.json").read_text())
pd.Series(class_histogram(ann)).sort_values().plot.barh(title="Boxes per class (train)")

In [ ]:
boxes = pd.DataFrame([a["bbox"] for a in ann["annotations"]], columns=["x", "y", "w", "h"])
boxes["area_px"] = boxes.w * boxes.h
# small boxes (< 32x32) are where motorcycles get lost in dense Bangkok traffic
(boxes.area_px < 32 * 32).mean()